In [70]:
import json
import numpy as np
import pandas as pd

from MATRIX.utils import get_data, get_metrics
from MATRIX.models import BaseSurvival, CoxRegression, DeepSurv, RandomSurvForest, DeepMultiTask, CoxRegressionWithTimeVarying, DeepTimeVarying

from remayn.result_set import ResultFolder

# Experiments

## Survival analysis (standard)

### Get data

In [2]:
X_train, y_train, X_validation, y_validation, X_test, y_test, feature_names, scaler = get_data(dataset_name="colon.csv")


- - - - colon.csv (csv) - - - -

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 911 entries, 0 to 910
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   event         911 non-null    int64  
 1   time          911 non-null    int64  
 2   num_age       911 non-null    int64  
 3   num_nodes     911 non-null    float64
 4   fac_rx        911 non-null    object 
 5   fac_sex       911 non-null    int64  
 6   fac_differ    911 non-null    object 
 7   fac_obstruct  911 non-null    int64  
 8   fac_perfor    911 non-null    int64  
 9   fac_adhere    911 non-null    int64  
 10  fac_node4     911 non-null    int64  
dtypes: float64(1), int64(8), object(2)
memory usage: 78.4+ KB

             event         time     num_age   num_nodes fac_rx     fac_sex  \
count   911.000000   911.000000  911.000000  911.000000    911  911.000000   
unique         NaN          NaN         NaN         NaN      3         NaN   
top 

### Instantiate models

In [3]:
modelCoxRegression = CoxRegression()
modelRandomSurvForest = RandomSurvForest(0)
modelDeepSurv = DeepSurv(X_train.shape[1])

### Fit and predict

In [4]:
modelCoxRegression = modelCoxRegression.fit(X_train, y_train)
modelRandomSurvForest = modelRandomSurvForest.fit(X_train, y_train)
modelDeepSurv = modelDeepSurv.fit(X_train, y_train)

survPredCoxRegression = modelCoxRegression.predict(X_test)
survPredRandomSurvForest = modelRandomSurvForest.predict(X_test)
survPredDeepSurv = modelDeepSurv.predict(X_test)

### Metrics

In [5]:
get_metrics([y_train, y_test], [survPredCoxRegression])

{'C-Index Harrel': 0.6115378193354484,
 'C-Index IPCW': 0.602885477942252,
 'Cumulative Dinamic AUC': [0.6197141965555563,
  0.6426197786856344,
  0.6419584638416529]}

In [6]:
get_metrics([y_train, y_test], [survPredRandomSurvForest])

{'C-Index Harrel': 0.6401736516947738,
 'C-Index IPCW': 0.6383489386036896,
 'Cumulative Dinamic AUC': [0.6430313586829167,
  0.6833542145552334,
  0.690041373973325]}

In [7]:
get_metrics([y_train, y_test], [survPredDeepSurv])

{'C-Index Harrel': 0.4948238437134747,
 'C-Index IPCW': 0.4930237285720696,
 'Cumulative Dinamic AUC': [0.5140764315693493,
  0.5141703073471825,
  0.446122910646115]}

### Survival / Cumulative Hazard Functions

In [ ]:
survival_function = modelCoxRegression.predict_survival_function(X=X_train, estimator_name="modelCoxRegression", dataset="colon.csv", seed=0, plot=False)

In [ ]:
cumulative_hazard_function = modelCoxRegression.predict_cumulative_hazard_function(X=X_train, estimator_name="modelCoxRegression", dataset="colon.csv", seed=0, plot=False)

### eXplainable Artificial Intelligence

In [10]:
modelCoxRegression.calculate_xai(X=X_train, estimator_name="modelCoxRegression", dataset="colon.csv", seed=0, feature_names=feature_names, background=50, plot=False)

PermutationExplainer explainer: 51it [00:13,  3.69it/s]                        


## Survival analysis (multitask)

### Get data

In [11]:
X_train, y_train, X_validation, y_validation, X_test, y_test, feature_names, scaler = get_data(dataset_name="colon.csv", to_multitask=True)


- - - - colon.csv (csv) - - - -

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 911 entries, 0 to 910
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   event         911 non-null    int64  
 1   time          911 non-null    int64  
 2   num_age       911 non-null    int64  
 3   num_nodes     911 non-null    float64
 4   fac_rx        911 non-null    object 
 5   fac_sex       911 non-null    int64  
 6   fac_differ    911 non-null    object 
 7   fac_obstruct  911 non-null    int64  
 8   fac_perfor    911 non-null    int64  
 9   fac_adhere    911 non-null    int64  
 10  fac_node4     911 non-null    int64  
dtypes: float64(1), int64(8), object(2)
memory usage: 78.4+ KB

             event         time     num_age   num_nodes fac_rx     fac_sex  \
count   911.000000   911.000000  911.000000  911.000000    911  911.000000   
unique         NaN          NaN         NaN         NaN      3         NaN   
top 

### Instantiate models

In [12]:
modelDeepMultiTask = DeepMultiTask(X_train.shape[1])

### Fit and predict

In [13]:
modelDeepMultiTask = modelDeepMultiTask.fit(X_train, y_train)
survPredDeepMultiTask, binPredDeepMultiTask = modelDeepMultiTask.predict(X_test)

### Metrics

In [14]:
get_metrics([y_train, y_test], [survPredDeepMultiTask, binPredDeepMultiTask])

{'C-Index Harrel': 0.48488896309901486,
 'C-Index IPCW': 0.49890225391941107,
 'Cumulative Dinamic AUC': [0.44667852245755924,
  0.47134928575899526,
  0.5389785919166161],
 'MAE': 0.48633879781420764,
 'AMAE': 0.48633879781420764,
 'MS': 0.1912568306010929,
 'CCR': 0.5136612021857924,
 'RECALL0': 0.3224043715846995,
 'RECALL1': 0.1912568306010929}

### Survival / Cumulative Hazard Functions

In [ ]:
survival_function = modelDeepMultiTask.predict_survival_function(X=X_train, estimator_name="modelDeepMultiTask", dataset="colon.csv", seed=0, plot=False)

In [ ]:
cumulative_hazard_function = modelDeepMultiTask.predict_cumulative_hazard_function(X=X_train, estimator_name="modelDeepMultiTask", dataset="colon.csv", seed=0, plot=False)

### eXplainable Artificial Intelligence

In [ ]:
modelDeepMultiTask.calculate_xai(X=X_train, estimator_name="modelDeepMultiTask", dataset="colon.csv", seed=0, feature_names=feature_names, background=50, plot=False)

## Survival analysis (time varying)

In [18]:
df = pd.read_csv("./MATRIX/datasets/colon.csv")

### Find splits dynamically

In [ ]:
splits = BaseSurvival.dinamic_discretise(y=df[["time", "event"]], dataset="colon", seed=0, plot=False)

In [20]:
df["identifier"] = df.index.values

### Transform to time varying format

In [21]:
df = BaseSurvival.to_time_dependent(dataframe=df, splits=splits, identifier="identifier", time="time", event="event")

In [22]:
df = BaseSurvival.to_time_varying(dataframe=df, identifier="identifier", time="time", event="event")

### Get data

In [23]:
X_train, y_train, X_validation, y_validation, X_test, y_test, feature_names, scaler = get_data(df=df)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4031 entries, 0 to 4030
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   identifier    4031 non-null   int64  
 1   num_age       4031 non-null   int64  
 2   num_nodes     4031 non-null   float64
 3   fac_rx        4031 non-null   object 
 4   fac_sex       4031 non-null   int64  
 5   fac_differ    4031 non-null   object 
 6   fac_obstruct  4031 non-null   int64  
 7   fac_perfor    4031 non-null   int64  
 8   fac_adhere    4031 non-null   int64  
 9   fac_node4     4031 non-null   int64  
 10  time_frame    4031 non-null   int64  
 11  days_risk     4031 non-null   float64
 12  event         4031 non-null   int64  
 13  time_start    4031 non-null   float64
 14  time_stop     4031 non-null   float64
dtypes: float64(4), int64(9), object(2)
memory usage: 472.5+ KB

         identifier      num_age    num_nodes   fac_rx      fac_sex  \
count   4031.000000  4031.

### Instantiate models

In [24]:
modelCoxRegressionWithTimeVarying = CoxRegressionWithTimeVarying()
modelDeepTimeVarying = DeepTimeVarying(X_train.shape[1])

### Fit and predict

In [25]:
modelCoxRegressionWithTimeVarying = modelCoxRegressionWithTimeVarying.fit(X_train, y_train)
modelDeepTimeVarying = modelDeepTimeVarying.fit(X_train, y_train)

survPredCoxRegressionWithTimeVarying = modelCoxRegressionWithTimeVarying.predict(X_test)
survPredDeepTimeVarying = modelDeepTimeVarying.predict(X_test)

### Metrics

In [26]:
get_metrics([y_train, y_test], [survPredCoxRegressionWithTimeVarying])

{'C-Index Harrel': 0.5757792250947033,
 'C-Index IPCW': 0.616212290202886,
 'Cumulative Dinamic AUC': [0.4548180147816044,
  0.5405238853225305,
  0.5638026366161393]}

In [27]:
get_metrics([y_train, y_test], [survPredDeepTimeVarying])

{'C-Index Harrel': 0.4227088456170665,
 'C-Index IPCW': 0.44245415171939695,
 'Cumulative Dinamic AUC': [0.3537515615743376,
  0.37517459937675524,
  0.38081060696089997]}

### Survival / Cumulative Hazard Functions

In [ ]:
survival_function = modelCoxRegressionWithTimeVarying.predict_survival_function(X=X_train, estimator_name="modelCoxRegressionWithTimeVarying", dataset="colon.csv", seed=0, plot=False)

In [ ]:
cumulative_hazard_function = modelCoxRegressionWithTimeVarying.predict_cumulative_hazard_function(X=X_train, estimator_name="modelCoxRegressionWithTimeVarying", dataset="colon.csv", seed=0, plot=False)

### eXplainable Artificial Intelligence

In [ ]:
modelCoxRegressionWithTimeVarying.calculate_xai(X=X_train, estimator_name="modelCoxRegressionWithTimeVarying", dataset="colon.csv", seed=0, feature_names=feature_names, background=50, plot=False)

# Results

In [68]:
rf = ResultFolder('./results')

In [90]:
for i, result in enumerate(rf):
    result.get_data()

    targets = [x for x in result.data_.targets[1]]
    predictions = [np.round(x, 3) for x in result.data_.predictions[0]]
    
    print(f"Result {i}:\n")
    print(f"    * Estimator: {result.config['estimator_name']} - Dataset: {result.config['dataset']} - Seed: {result.config['random_state']}")
    print(f"    * Best Params: {result.data_.best_params}")
    print()
    print(f"    * Targets: {targets[:5]}")
    print(f"    * Predictions: {predictions[:5]}")
    
    break

Result 0:

    * Estimator: CoxRegression - Dataset: rhc.csv - Seed: 20
    * Best Params: {'ties': 'efron', 'n_iter': 100, 'alpha': 1.0}

    * Targets: [(True, 876.), (True, 813.), (True, 804.), (True, 752.), (True, 683.)]
    * Predictions: [-0.537, 1.437, 1.765, -0.535, -1.054]
